#**CRM MCP Server — Project 3**

A secure, production-style Model Context Protocol (MCP) server for AI-powered CRM interactions.
Built with Python, MCP, SQLite, and pytest, featuring 6 validated tools, 2 resources, and 1 prompt for controlled customer search, interaction management, and pipeline analysis. The project uses parameterized SQL, explicit business operations, automated testing, and synthetic CRM data, with no API keys or paid services required. Designed to run offline in Google Colab and integrate with AI agents safely.

In [1]:
from google.colab import files
import zipfile
import os

uploaded = files.upload()

zip_name = next(iter(uploaded))

extract_dir = "/content/crm-mcp-server"

if os.path.exists(extract_dir):
    import shutil
    shutil.rmtree(extract_dir)

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print("✅ Project extracted to:", extract_dir)

Saving files (2).zip to files (2).zip
✅ Project extracted to: /content/crm-mcp-server


In [2]:
!find /content -maxdepth 3 -type f | sort

/content/.config/active_config
/content/.config/config_sentinel
/content/.config/configurations/config_default
/content/.config/default_configs.db
/content/.config/gce
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
/content/.config/.last_opt_in_prompt.yaml
/content/.config/.last_survey_prompt.yaml
/content/.config/.last_update_check.json
/content/crm-mcp-server/client_demo.py
/content/crm-mcp-server/database.py
/content/crm-mcp-server/.gitignore
/content/crm-mcp-server/README.md
/content/crm-mcp-server/requirements.txt
/content/crm-mcp-server/seed_database.py
/content/crm-mcp-server/server.py
/content/crm-mcp-server/test_server.py
/content/files (2).zip
/content/sample_data/anscombe.json
/content/sample_data/california_housing_test.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/mnist_test.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/README.md


In [3]:
%cd /content/crm-mcp-server

/content/crm-mcp-server


In [4]:
!ls

client_demo.py	README.md	  seed_database.py  test_server.py
database.py	requirements.txt  server.py


In [5]:
!pip install -q "mcp[cli]" pytest

In [6]:
!pip show mcp

Name: mcp
Version: 2.1.0
Summary: Model Context Protocol SDK
Home-page: https://modelcontextprotocol.io
Author: Model Context Protocol a Series of LF Projects, LLC.
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: anyio, httpx2, jsonschema, mcp-types, opentelemetry-api, pydantic, pyjwt, python-multipart, sse-starlette, starlette, typing-extensions, typing-inspection, uvicorn
Required-by: 


In [7]:
!python seed_database.py

CRM database initialized.

Customers: 20
Interactions: 40


In [8]:
!ls -lh

total 116K
-rw-r--r-- 1 root root 4.3K Aug 25 10:54 client_demo.py
-rw-r--r-- 1 root root  32K Aug 25 10:55 crm.db
-rw-r--r-- 1 root root  12K Aug 25 10:54 database.py
drwxr-xr-x 2 root root 4.0K Aug 25 10:55 __pycache__
-rw-r--r-- 1 root root  13K Aug 25 10:54 README.md
-rw-r--r-- 1 root root   16 Aug 25 10:54 requirements.txt
-rw-r--r-- 1 root root  12K Aug 25 10:54 seed_database.py
-rw-r--r-- 1 root root  14K Aug 25 10:54 server.py
-rw-r--r-- 1 root root  11K Aug 25 10:54 test_server.py


In [9]:
!pytest -q

......................                                                   [100%]
22 passed in 10.11s


In [10]:
!python client_demo.py


1. Tool discovery
- find_customers
- get_customer
- update_customer_status
- add_interaction
- get_interactions
- customer_pipeline_summary

2. find_customers(query='Nimbus')
{
  "count": 1,
  "customers": [
    {
      "id": 1,
      "name": "Aarav Sharma",
      "email": "aarav.sharma@nimbuslogix.com",
      "company": "Nimbus Logix",
      "country": "India",
      "industry": "Logistics",
      "status": "customer",
      "created_at": "2025-01-05T09:00:00+00:00",
      "last_contacted_at": "2025-06-10T11:30:00+00:00"
    }
  ]
}

3. get_customer(customer_id=1)
{
  "customer": {
    "id": 1,
    "name": "Aarav Sharma",
    "email": "aarav.sharma@nimbuslogix.com",
    "company": "Nimbus Logix",
    "country": "India",
    "industry": "Logistics",
    "status": "customer",
    "created_at": "2025-01-05T09:00:00+00:00",
    "last_contacted_at": "2025-06-10T11:30:00+00:00"
  },
  "interaction_count": 3,
  "recent_interactions": [
    {
      "id": 1,
      "customer_id": 1,
      "int

In [11]:
import sqlite3

conn = sqlite3.connect("crm.db")

tables = conn.execute("""
SELECT name
FROM sqlite_master
WHERE type='table'
""").fetchall()

print("Tables:")
for table in tables:
    print(" -", table[0])

conn.close()

Tables:
 - customers
 - sqlite_sequence
 - interactions


In [12]:
import sqlite3

conn = sqlite3.connect("crm.db")

print("CUSTOMERS")
print("=" * 50)

rows = conn.execute("""
SELECT id, name, email, company, status
FROM customers
ORDER BY id
""").fetchall()

for row in rows:
    print(row)

conn.close()

CUSTOMERS
(1, 'Aarav Sharma', 'aarav.sharma@nimbuslogix.com', 'Nimbus Logix', 'customer')
(2, 'Meera Iyer', 'meera.iyer@brightfin.com', 'BrightFin', 'customer')
(3, 'Rohan Verma', 'rohan.verma@cropsense.io', 'CropSense', 'prospect')
(4, 'Ananya Rao', 'ananya.rao@healwell.com', 'HealWell', 'prospect')
(5, 'Kabir Malhotra', 'kabir.malhotra@edulaunch.com', 'EduLaunch', 'prospect')
(6, 'Diya Patel', 'diya.patel@greenwatt.com', 'GreenWatt Energy', 'customer')
(7, 'Vivaan Nair', 'vivaan.nair@stackforge.dev', 'StackForge', 'customer')
(8, 'Ishita Desai', 'ishita.desai@retailyze.com', 'Retailyze', 'inactive')
(9, 'Arjun Kapoor', 'arjun.kapoor@medicore.com', 'MediCore', 'lead')
(10, 'Sanya Chatterjee', 'sanya.chatterjee@fleetwise.io', 'FleetWise', 'prospect')
(11, 'Dev Joshi', 'dev.joshi@quantifyhr.com', 'QuantifyHR', 'customer')
(12, 'Priya Menon', 'priya.menon@sunrisecapital.com', 'Sunrise Capital', 'inactive')
(13, 'Karan Bhatt', 'karan.bhatt@urbanfresh.com', 'UrbanFresh', 'lead')
(14, 'Neha

In [13]:
import sqlite3

conn = sqlite3.connect("crm.db")

print("INTERACTIONS")
print("=" * 50)

rows = conn.execute("""
SELECT id, customer_id, interaction_type, note
FROM interactions
ORDER BY id
LIMIT 10
""").fetchall()

for row in rows:
    print(row)

conn.close()

INTERACTIONS
(1, 1, 'call', 'Discussed renewal terms for the logistics tracking module.')
(2, 1, 'email', 'Sent updated pricing sheet after the renewal call.')
(3, 1, 'meeting', 'Quarterly business review with the ops team.')
(4, 2, 'demo', 'Walked through the fraud-detection dashboard with the compliance lead.')
(5, 2, 'call', 'Follow-up call to answer questions about API rate limits.')
(6, 3, 'meeting', 'In-person meeting to scope a pilot for soil-sensor integration.')
(7, 3, 'email', 'Sent pilot proposal document for review.')
(8, 5, 'demo', 'Product demo focused on the course-authoring tools.')
(9, 5, 'note', 'Prospect mentioned budget approval expected next quarter.')
(10, 6, 'call', 'Annual contract review, confirmed continued usage.')
